# Exercise 1: Linear Regression on mtcars

Car consumptions are measured in MPGs (miles per gallon): higher MPG values mean that the car is able to do more miles with one gallon of fuel.

To determine the car features having an effect on consumption, we use a linear regression model with all predictors.

**Original R code:**
```r
M <- lm(mpg~., data=mtcars)
summary(M)
```

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Load the mtcars dataset (first column is car name, used as index)
mtcars = pd.read_csv('../mtcars.csv', index_col=0)
print(mtcars.shape)
mtcars.head()

In [ ]:
# Fit OLS regression: mpg ~ all other predictors
y = mtcars['mpg']
X = mtcars.drop(columns=['mpg'])
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()
print(model.summary())

## Question 1

**Which predictors seem to have an effect on mpg?**

Make your criterion explicit.

### Answer:

We use p-value < 0.05 as the criterion for statistical significance at the 5% level.

In [ ]:
# Show predictors with p-value < 0.05 (excluding the intercept/const)
pvalues = model.pvalues.drop('const')
significant = pvalues[pvalues < 0.05]

print("Significant predictors (p < 0.05):")
print(significant.sort_values())
print()
print("All p-values:")
print(pvalues.sort_values())

## Question 2

**Provide a 95% confidence interval for the `carb` parameter.**

**What can you deduce from the fact that 0 is in the confidence interval?**

**What does the quantity in column `Pr(>|t|)` represent?**

### Answer:

The column `Pr(>|t|)` (p-value for the t-test) represents the probability of observing a t-statistic as extreme as or more extreme than the one computed, under the null hypothesis H0: beta_j = 0. Formally:

$$p = P(|T| \geq |t_{obs}|) = 2 \cdot P\left(T \geq \frac{|\hat{\beta}_j|}{\hat{\sigma}_{\hat{\beta}_j}}\right)$$

where $T \sim t_{n-p-1}$ under H0.

If 0 is contained in the 95% CI for `carb`, we cannot reject H0: beta_carb = 0 at the 5% level, meaning we have no statistically significant evidence that `carb` has a linear effect on `mpg` when all other predictors are included.

In [ ]:
# 95% confidence interval for all coefficients
conf_int = model.conf_int(alpha=0.05)
conf_int.columns = ['2.5%', '97.5%']

print("95% Confidence Interval for carb:")
print(conf_int.loc['carb'])
print()
print("Full 95% CI table:")
print(conf_int)

## Question 3

**What does the overall F-test p-value represent? Does it seem in discordance with Q1?**

### Answer:

The F-test p-value (shown as `Prob (F-statistic)` in the summary) tests the null hypothesis that **all** regression coefficients are simultaneously zero:

$$H_0: \beta_1 = \beta_2 = \cdots = \beta_p = 0$$

The F-statistic is:
$$F = \frac{(SSR/p)}{(SSE/(n-p-1))} = \frac{\text{Explained variance per predictor}}{\text{Residual variance per df}}$$

A very small p-value (e.g., 3.79e-07) means we strongly reject H0: the model as a whole explains a significant portion of the variance in mpg.

**Apparent discordance with Q1:** In Q1, none or very few individual predictors appear significant at 5%, yet the overall model is highly significant. This is a classic symptom of **multicollinearity**: the predictors are highly correlated with each other (e.g., `cyl`, `disp`, `hp`, `wt` are all related to engine size). When predictors are correlated, their individual standard errors inflate, making individual t-tests non-significant, while the collective explanatory power measured by the F-test remains high.

## Question 4

**What strategy could be used to obtain a linear regression model with all predictors significant at 5%?**

### Answer:

A common strategy is **backward elimination**:
1. Start with the full model containing all predictors.
2. At each step, remove the predictor with the highest p-value (least significant), provided it is above the threshold alpha = 0.05.
3. Refit the model and repeat until all remaining predictors are significant at 5%.

This is feasible here since n=32 > p=10. However, due to multicollinearity among the predictors, the coefficients and their significance will change as predictors are removed. The procedure may be sensitive to the order of elimination and might not yield a unique result. An alternative criterion-based approach (AIC/BIC minimisation) is generally preferred.

## Question 5

**Describe and run some algorithm to find the best model in some sense.**

### Answer:

We use **forward stepwise selection based on AIC (Akaike Information Criterion)**:

$$AIC = 2k - 2\ln(\hat{L})$$

where $k$ is the number of parameters and $\hat{L}$ is the maximised likelihood. AIC penalises model complexity while rewarding goodness of fit, offering a principled trade-off.

**Algorithm:**
1. Start with a model containing only the intercept.
2. At each step, try adding each remaining predictor one at a time and compute the AIC of the resulting model.
3. Add the predictor that results in the greatest AIC decrease.
4. Stop when no addition decreases the AIC.

In [ ]:
# Forward stepwise selection by AIC

def forward_stepwise_aic(y, X):
    """Forward stepwise variable selection minimising AIC."""
    remaining = list(X.columns)
    selected = []
    current_aic = sm.OLS(y, sm.add_constant(pd.DataFrame(index=X.index))).fit().aic

    print(f"Starting AIC (intercept only): {current_aic:.4f}\n")

    while remaining:
        best_aic = current_aic
        best_feature = None

        for feature in remaining:
            candidate = selected + [feature]
            X_cand = sm.add_constant(X[candidate])
            aic = sm.OLS(y, X_cand).fit().aic
            if aic < best_aic:
                best_aic = aic
                best_feature = feature

        if best_feature is None:
            print("No further improvement. Stopping.")
            break

        selected.append(best_feature)
        remaining.remove(best_feature)
        current_aic = best_aic
        print(f"Added: {best_feature:12s}  |  AIC: {current_aic:.4f}  |  Selected so far: {selected}")

    return selected

X_features = mtcars.drop(columns=['mpg'])
best_features = forward_stepwise_aic(y, X_features)

In [ ]:
# Final model summary
X_best = sm.add_constant(X_features[best_features])
final_model = sm.OLS(y, X_best).fit()
print("Final model selected by forward AIC:")
print(final_model.summary())